# Measure tissue thickness

Variable tissue thickness means a fraction of every FOV's z-stack has no real tissue signal at all -- wasted imaging time. This notebook maps, per FOV, where (in z, µm) real tissue signal actually starts and ends, using one already-finished round (default: `cells`).

Procedure:
1. Resolve the target round's frame table and `CHANNEL_NM`'s z-steps.
2. For every FOV, read every z-plane of `CHANNEL_NM` (still far fewer than the round's full multi-color frame count) and build an EXACT, bin-width-1 histogram of each frame -- a true Counter over observed pixel intensities (`analysis.fov.compute_channel_counters`, stored sparsely via `numpy.unique`, cached per FOV). Exact per-intensity counts mean every later step (reference-frame selection, threshold estimation, the per-z true-pixel-count profile) is derived from this one cached read, without recomputing or re-reading pixels.
3. Across every FOV and z, find the single frame with the **highest mean intensity** and use it as the reference frame for threshold estimation -- not a fixed z pooled across FOVs (an earlier version's approach, which broke down once it became clear some FOVs are blank at that fixed z; see step 4). Display that frame's image, plus a log-scale histogram (drives the automatic threshold estimate, the valley between its two most prominent peaks -- reuses `acquisition.mosaic._estimate_bimodal_threshold`'s exact peak-finding) and a linear-scale histogram from its minimum value to a percentile cutoff -- review both and override `THRESHOLD` manually if the auto-estimate looks wrong.
4. For every FOV, derive its true-pixel-count (NTP) profile directly from its cached Counter (no further disk read). Report the shallowest (`z_first_um`) and deepest (`z_last_um`) z with signal (NTP > `NTP_THRESHOLD`), plus `is_contiguous` (whether signal held continuously in between). Both boundaries matter: some FOVs are blank at the top of the imaged range and only pick up signal partway down, not just "signal that eventually stops".
5. Lay every FOV's `z_first_um`/`z_last_um` out on its stage-position grid and plot as heatmaps.

Figures and results are saved under `SAMPLE_DIR/analysis/figures/` and `SAMPLE_DIR/analysis/` respectively, in addition to being shown inline.

Runs anywhere the standard `SAMPLE_DIR/{data,metadata,positions,analysis}` layout is reachable, including a cluster node (same convention as `05_batch_sample_review.ipynb`/`07_cluster_submit_analysis.ipynb`). Step 2's backfill loop is intentionally sequential, not process-pool-parallelized: on a shared SLURM node, `os.cpu_count()` reports the node's total core count, not this job's actual memory allocation, so sizing a worker pool off it (`config.resolved_n_workers`) can spawn far more workers than the job's real memory allows -- each holding a stack in memory at once -- and get OOM-killed (`BrokenProcessPool`). Reading only `CHANNEL_NM`'s frames (not the whole multi-color stack) keeps the sequential version fast without a pool.

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config      import ExperimentConfig
from MERci.common.metadata    import ExperimentMetadata
from MERci.progress           import ProgressTracker
from MERci.progress_display   import ProgressReporter
from MERci.common.io          import iter_image_frames
from MERci.analysis.fov       import (
    compute_channel_counters, save_channel_counters, load_channel_counters,
    counter_mean, counter_percentile, rebin_counter, ntp_profile_from_counters,
)
from MERci.acquisition.configs import find_frame_table_for_hal_config
from MERci.acquisition.mosaic  import _estimate_bimodal_threshold

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME  = SAMPLE_DIR.name
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

# Which round to use -- by imaging_type (default "cells"), or set ROUND_ID directly to override.
ROUND_IMAGING_TYPE = "cells"
ROUND_ID            = None

# Channel to measure tissue depth for.
CHANNEL_NM = 405.0

# A z-plane still counts as "has tissue" if its true-pixel count (NTP) exceeds this.
NTP_THRESHOLD = 1

# Heatmap color scale upper bound (micron).
MAX_Z_COLORMAP = 70.0

# Display-histogram resolution (section 5) -- these are re-binned on demand from the
# exact per-intensity Counter, so changing this never requires recomputing anything.
DISPLAY_HIST_BINS      = 200
LINEAR_HIST_PERCENTILE = 99.0   # upper bound of the linear-scale display histogram

# Binarization intensity threshold; None = auto-estimate from the single frame with
# the highest mean intensity across every FOV (section 5) -- review that plot before
# trusting the estimate on a new experiment.
THRESHOLD = None

print(f"Sample name        : {SAMPLE_NAME}")
print(f"Round imaging type : {ROUND_IMAGING_TYPE}  (ROUND_ID override: {ROUND_ID})")
print(f"Channel            : {CHANNEL_NM} nm")

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{SAMPLE_NAME}.txt",
    image_suffix   = IMAGE_SUFFIX,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

figures_dir = config.analysis_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# This notebook's own cache -- deliberately NOT tracker.histogram_path()'s canonical
# location. FOVScheduler's own per-frame histograms are fixed-width (lossy) 512-bin
# ones, which can't be turned back into an EXACT per-intensity Counter, so unlike
# earlier versions of this notebook there is no shortcut reuse of that canonical
# cache at all here -- every FOV's channel Counters are computed by, and cached
# under, this notebook alone.
tissue_cache_dir     = config.analysis_dir / "tissue_thickness_cache"
channel_counters_dir = tissue_cache_dir / "channel_counters"
channel_counters_dir.mkdir(parents=True, exist_ok=True)

print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")
print(f"Figures: {figures_dir}")
print(f"Cache  : {tissue_cache_dir}")

## 3 — Resolve the target round and its frame table

In [ ]:
def resolve_round_id(meta, imaging_type):
    """First round_id whose series carry the given imaging_type."""
    for rid in meta.valid_round_ids():
        if any((s.imaging_type or "").strip().lower() == imaging_type.strip().lower()
               for s in meta.series_for_round(rid)):
            return rid
    raise ValueError(f"No round found with imaging_type={imaging_type!r}")


def load_round_frame_table(round_id, config, meta):
    """Frame table (columns color/channel/z, 0-based frame-index rows) for round_id's HAL config."""
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        hal_path = Path(config.settings_dir) / s.hal_config
        ft_path  = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
        if ft_path and ft_path.exists():
            return pd.read_csv(ft_path, index_col=0)
    raise FileNotFoundError(f"No frame table found for round {round_id}")


target_round_id = ROUND_ID if ROUND_ID is not None else resolve_round_id(meta, ROUND_IMAGING_TYPE)
if not meta.round_fully_written(target_round_id):
    print(f"WARNING: round {target_round_id} is not yet fully written on disk -- "
          f"results below will be based on a partial FOV set.")

frame_table = load_round_frame_table(target_round_id, config, meta)

channel_frames = (
    frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)]
    .sort_values("z")
)
if channel_frames.empty:
    raise ValueError(f"No frames found for channel {CHANNEL_NM} nm in round {target_round_id}'s frame table.")

z_frame_indices = list(zip(channel_frames.index.tolist(), channel_frames["z"].tolist()))

print(f"Target round : {target_round_id}")
print(f"Channel {CHANNEL_NM} nm has {len(z_frame_indices)} z-step(s) in this round's frame table.")

## 4 — Compute (or load cached) exact per-z Counter histograms for `CHANNEL_NM`

For every FOV, reads every z-plane of `CHANNEL_NM` (still far fewer than the round's
full multi-color frame count) and builds an EXACT, bin-width-1 histogram of each
frame -- a true Counter over observed pixel intensities (`analysis.fov.
compute_channel_counters`, stored sparsely: only intensity values that actually
occur, via `numpy.unique`), not a fixed-bin-count histogram. Having every
intensity's exact count means the reference-frame selection (section 5), the
threshold-estimation display histograms (section 5), and the per-z true-pixel-count
profile (section 6) can ALL be derived from this ONE cached read, without
re-reading pixels or recomputing anything.

No fallback to an existing full per-frame histogram here (unlike earlier versions
of this notebook): `FOVScheduler`'s own histograms are fixed-width (lossy) 512-bin
ones, which can't be turned back into an exact per-intensity Counter -- every FOV's
Counters are computed by, and cached under, this notebook alone
(`analysis/tissue_thickness_cache/channel_counters/`).

In [ ]:
def channel_counters_path(fpath):
    return channel_counters_dir / f"{Path(fpath).stem}_counters.npz"


files = meta.files_for_round(target_round_id)
print(f"Round {target_round_id}: {len(files)} FOV file(s) expected.")

channel_counters = {}   # fov_id -> compute_channel_counters()-shaped dict
from_own_cache, to_compute = [], []
n_missing_on_disk = 0

for fpath in files:
    if channel_counters_path(fpath).exists():
        from_own_cache.append(fpath)
    elif fpath.exists():
        to_compute.append(fpath)
    else:
        n_missing_on_disk += 1

for fpath in from_own_cache:
    channel_counters[meta.fov_id_of_file(fpath)] = load_channel_counters(channel_counters_path(fpath))
print(f"{len(from_own_cache)} channel Counter(s) already cached -- loaded directly.")

n_computed = 0
if to_compute:
    print(f"Computing {len(to_compute)} missing channel Counter(s) sequentially "
          f"(all {len(z_frame_indices)} z-step(s) of channel {CHANNEL_NM:.0f} nm per FOV).")
    reporter = ProgressReporter(total=len(to_compute), label="Computing channel Counters")
    for fpath in reporter.wrap(to_compute):
        counters = compute_channel_counters(
            fpath, z_frame_indices,
            frame_width=config.frame_width, frame_height=config.frame_height,
        )
        save_channel_counters(channel_counters_path(fpath), counters)
        channel_counters[meta.fov_id_of_file(fpath)] = counters
        n_computed += 1

print(f"Channel Counters ready for {len(channel_counters)} / {len(files)} FOVs "
      f"({n_computed} newly computed this run, {n_missing_on_disk} not yet written on disk).")

## 5 — Reference frame (highest mean intensity) and threshold estimation

Across every FOV and every z, finds the single frame with the highest mean
intensity -- the most tissue-dense frame available anywhere in the round, rather
than pooling one fixed z across every FOV (an earlier version's approach, which
produced a poor, non-bimodal pooled histogram since that fixed z is blank for some
FOVs -- see section 6's `z_first_um`). Displays that frame's image directly for a
visual sanity check, plus two views of its exact histogram: log-scale (used for the
automatic threshold estimate) and linear-scale from its minimum value to the
`LINEAR_HIST_PERCENTILE`-th percentile. Review both before trusting `THRESHOLD` --
if the estimate looks wrong (or no threshold is found at all), set `THRESHOLD` by
hand above and re-run from here.

In [ ]:
best = None   # (mean, fov_id, pos_in_z, frame_idx, z_um)
n_frames_considered = 0
for fov_id, counters in channel_counters.items():
    for pos, (values, counts) in enumerate(zip(counters["values_per_z"], counters["counts_per_z"])):
        n_frames_considered += 1
        mean = counter_mean(values, counts)
        if best is None or mean > best[0]:
            best = (mean, fov_id, pos, int(counters["frame_indices"][pos]), float(counters["z_um"][pos]))

best_mean, best_fov_id, best_pos, best_frame_idx, best_z_um = best
best_values = channel_counters[best_fov_id]["values_per_z"][best_pos]
best_counts = channel_counters[best_fov_id]["counts_per_z"][best_pos]

print(f"Reference frame: FOV {best_fov_id}, frame_idx={best_frame_idx}, z={best_z_um:.2f} um "
      f"(mean intensity {best_mean:.0f}, highest of {n_frames_considered} FOV x z combinations)")

# Counters have no spatial information -- re-read just this one frame to display it.
best_fpath = next(f for f in files if meta.fov_id_of_file(f) == best_fov_id)
best_frame = next(frame for _, frame in iter_image_frames(
    best_fpath, [best_frame_idx], frame_width=config.frame_width, frame_height=config.frame_height,
))

fig_img, ax_img = plt.subplots(figsize=(5, 5))
im = ax_img.imshow(best_frame, cmap="gray")
ax_img.set_title(f"Reference frame -- FOV {best_fov_id}, z={best_z_um:.1f} um")
fig_img.colorbar(im, ax=ax_img, fraction=0.046, pad=0.04)
fig_img.tight_layout()
fig_img.savefig(figures_dir / f"tissue_thickness_reference_frame_round{target_round_id}.png", dpi=150)
plt.show()

# Log-scale view (drives the automatic threshold estimate) + linear-scale view
# (min -> LINEAR_HIST_PERCENTILE), both re-binned on demand from the exact Counter --
# no raw pixel re-read for either.
min_value = float(best_values.min())
max_value = float(best_values.max())
pct_value = counter_percentile(best_values, best_counts, LINEAR_HIST_PERCENTILE)

log_edges       = np.logspace(np.log10(max(min_value, 1)), np.log10(max_value), DISPLAY_HIST_BINS + 1)
log_counts      = rebin_counter(best_values, best_counts, log_edges)
log_bin_centers = np.sqrt(log_edges[:-1] * log_edges[1:])   # geometric mean = correct center in log space

linear_edges       = np.linspace(min_value, pct_value, DISPLAY_HIST_BINS + 1)
linear_counts       = rebin_counter(best_values, best_counts, linear_edges)
linear_bin_centers  = 0.5 * (linear_edges[:-1] + linear_edges[1:])

# Reuses mosaic.py's peak-finding valley estimator verbatim on the log-scale view: it
# expects LOG-space bin centers and returns 10**valley (linear units).
estimated_threshold = _estimate_bimodal_threshold(np.log10(log_bin_centers), log_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(log_bin_centers, log_counts, "-", color="black", lw=1.2)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel(f"Intensity  (channel {CHANNEL_NM:.0f} nm)")
axes[0].set_ylabel("Pixel count")
axes[0].set_title("Log-scale (full range)")

axes[1].plot(linear_bin_centers, linear_counts, "-", color="black", lw=1.2)
axes[1].set_xlabel(f"Intensity  (channel {CHANNEL_NM:.0f} nm)")
axes[1].set_ylabel("Pixel count")
axes[1].set_title(f"Linear-scale (min={min_value:.0f} -> p{LINEAR_HIST_PERCENTILE:.0f}={pct_value:.0f})")

if estimated_threshold is not None:
    for ax in axes:
        ax.axvline(estimated_threshold, color="crimson", linestyle="--", lw=1.5,
                   label=f"estimated threshold = {estimated_threshold:.0f}")
        ax.legend()

fig.suptitle(f"Round {target_round_id} -- reference-frame histogram (FOV {best_fov_id}, z={best_z_um:.1f} um)")
fig.tight_layout()
fig.savefig(figures_dir / f"tissue_thickness_histogram_round{target_round_id}.png", dpi=150)
plt.show()

if THRESHOLD is None:
    THRESHOLD = estimated_threshold
    if THRESHOLD is None:
        raise ValueError(
            "Could not auto-estimate a threshold (not clearly bimodal) -- set THRESHOLD manually above."
        )
print(f"Using THRESHOLD = {THRESHOLD:.0f}")

## 6 — Per-FOV: true-pixel-count (NTP) profile, derived from cached Counters

Purely in-memory now: every FOV's exact per-z Counter is already cached/loaded from
section 4, so deriving each z's true-pixel count against `THRESHOLD`
(`analysis.fov.ntp_profile_from_counters`) needs no further disk read at all.
Reports both `z_first_um`/`z_last_um` (shallowest/deepest z with signal) and
`is_contiguous` (`False` if signal turned off and back on somewhere in between --
debris, folded tissue, noise) -- some FOVs are blank at the top of the imaged range
and only pick up tissue signal partway down, so both boundaries matter, not just
"signal that eventually stops".

In [ ]:
results = []
for fov_id, counters in channel_counters.items():
    profile = ntp_profile_from_counters(counters, THRESHOLD, NTP_THRESHOLD)
    results.append({
        "fov_id":        fov_id,
        "z_first_um":    profile["z_first_um"],
        "z_last_um":     profile["z_last_um"],
        "is_contiguous": profile["is_contiguous"],
        "x_um":          meta.fovs[fov_id].position[0],
        "y_um":          meta.fovs[fov_id].position[1],
    })

results_df = pd.DataFrame(results)
n_no_signal      = results_df["z_last_um"].isna().sum()
n_not_contiguous = (~results_df["is_contiguous"]).sum()
print(f"{len(results_df)} FOV(s) measured; {n_no_signal} had no z-plane above NTP_THRESHOLD at all; "
      f"{n_not_contiguous} had signal turn off and back on somewhere in between (is_contiguous=False).")
print("z_first_um:")
print(results_df["z_first_um"].describe())
print("z_last_um:")
print(results_df["z_last_um"].describe())

## 7 — Tissue-extent heatmaps across the FOV grid

Two panels sharing one color scale: where tissue signal **starts** (`z_first_um` --
a shallow-imaged range wasted before signal appears would show up here as bright
patches) and where it **ends** (`z_last_um`, the original "how deep does tissue go"
question).

In [ ]:
def positions_to_grid_indices(fov_ids, meta):
    """Stage (x, y) positions -> integer (x_idx, y_idx) grid indices (same approach as
    04_view_intensity_stats.ipynb's heatmap: round to the nearest integer micron, then
    rank each axis's unique values -- robust to float imprecision on a regular grid)."""
    xs = np.array([round(meta.fovs[f].position[0]) for f in fov_ids])
    ys = np.array([round(meta.fovs[f].position[1]) for f in fov_ids])
    unique_xs = np.sort(np.unique(xs))
    unique_ys = np.sort(np.unique(ys))
    x_rank = {v: i for i, v in enumerate(unique_xs)}
    y_rank = {v: i for i, v in enumerate(unique_ys)}
    return {f: (x_rank[xs[i]], y_rank[ys[i]]) for i, f in enumerate(fov_ids)}


fov_ids = results_df["fov_id"].tolist()
grid    = positions_to_grid_indices(fov_ids, meta)
n_x     = max(xi for xi, _ in grid.values()) + 1
n_y     = max(yi for _, yi in grid.values()) + 1


def build_matrix(column):
    matrix = np.full((n_y, n_x), np.nan)
    for _, row in results_df.iterrows():
        xi, yi = grid[row["fov_id"]]
        if pd.notna(row[column]):
            matrix[yi, xi] = row[column]
    return matrix


fig, axes = plt.subplots(1, 2, figsize=(max(9, n_x * 0.7 + 3), max(4, n_y * 0.4 + 1.5)))
for ax, column, title in zip(
    axes,
    ("z_first_um", "z_last_um"),
    ("z_first -- signal starts", "z_last -- signal ends"),
):
    im = ax.imshow(build_matrix(column), cmap="viridis", origin="upper", vmin=0, vmax=MAX_Z_COLORMAP)
    ax.set_title(title)
    ax.set_xlabel("X grid index  (increasing stage X →)")
    ax.set_ylabel("Y grid index  (increasing stage Y ↓)")
cbar = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.04)
cbar.set_label("z (um)")
fig.suptitle(f"Round {target_round_id} -- tissue extent map ({len(fov_ids)} FOVs)")

fig.savefig(figures_dir / f"tissue_thickness_heatmap_round{target_round_id}.png", dpi=150)
plt.show()

results_csv = config.analysis_dir / f"tissue_thickness_round{target_round_id}.csv"
results_df.to_csv(results_csv, index=False)
print(f"Saved: {results_csv}")